# Foundations I: Learning as compression

Companion to `slides/slides_01.md`.

Three live demos:

* **D1 — Memorization vs. generalization:** a lookup table against a fitted curve.
* **D2 — Entropy and compression:** structured text, natural text, and random bytes.
* **D3 — Model selection as compression:** polynomial degree vs. training error, test error, and MDL.

A short bridge at the end shows that *finding* the model is an optimization problem (next notebooks).

In [ ]:
import math
import os
import zlib
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from numpy.polynomial import Polynomial

rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 4)

## D1 — Memorization vs. generalization

The data come from an unknown function plus noise:

$$y = f(x) + \varepsilon, \qquad f(x) = \sin(2\pi x), \quad \varepsilon \sim \mathcal{N}(0, 0.3^2)$$

Two "learners":

1. **Lookup table:** stores every training pair. For a new $x$ it copies the answer of the closest stored $x$.
2. **Model:** a degree-3 polynomial, i.e. only **4 numbers**.

In [ ]:
SIGMA = 0.3


def f(x):
    return np.sin(2 * np.pi * x)


def sample(n):
    x = np.sort(rng.uniform(0, 1, n))
    return x, f(x) + rng.normal(0, SIGMA, n)


x_train, y_train = sample(50)
x_test, y_test = sample(1000)


def lookup_table(x_query):
    idx = np.abs(x_query[:, None] - x_train[None, :]).argmin(axis=1)
    return y_train[idx]


model = Polynomial.fit(x_train, y_train, deg=3)


def mse(y, y_hat):
    return float(np.mean((y - y_hat) ** 2))


print(f"{'':15s}{'train MSE':>12s}{'test MSE':>12s}{'numbers stored':>16s}")
print(
    f"{'lookup table':15s}{mse(y_train, lookup_table(x_train)):12.4f}{mse(y_test, lookup_table(x_test)):12.4f}{2 * len(x_train):16d}"
)
print(f"{'cubic model':15s}{mse(y_train, model(x_train)):12.4f}{mse(y_test, model(x_test)):12.4f}{4:16d}")
print(f"\nIrreducible noise (sigma^2) = {SIGMA**2:.4f}")

In [ ]:
grid = np.linspace(0, 1, 500)
fig, ax = plt.subplots()
ax.scatter(x_train, y_train, color="k", zorder=3, label="training data")
ax.plot(grid, f(grid), "g--", label="true f(x)")
ax.step(grid, lookup_table(grid), where="mid", color="tab:red", label="lookup table")
ax.plot(grid, model(grid), color="tab:blue", lw=2, label="cubic model")
ax.set(xlabel="x", ylabel="y", title="Memorizing the noise vs. compressing the pattern")
ax.legend()
plt.show()

**Takeaway:** the lookup table is perfect on data it has seen and worse on new data, because it memorized the noise.
The cubic model stores far fewer numbers, and its test error is close to the noise floor $\sigma^2$.
**Compressing the data forced it to keep the pattern and discard the noise.**

## D2 — Entropy and compression

Shannon entropy (order-0, i.e. symbol frequencies only):

$$H(X) = -\sum_{x} p(x)\log_2 p(x) \quad \text{bits per symbol}$$

We compare it with what a real compressor (`zlib`, the algorithm behind gzip) achieves.

In [ ]:
TEXT = (
    "Machine learning is the study of algorithms that improve their performance at some task through experience. "
    "A learning algorithm receives a finite set of examples and must produce a model that behaves well on examples "
    "it has never seen before. This is only possible when the examples share some regularity: a pattern that can be "
    "described more briefly than the examples themselves. If the data were pure noise, there would be nothing to "
    "learn, because no description shorter than the data itself would exist. The art of learning is therefore the "
    "art of finding short descriptions, of separating what is structure from what is accident. Simple models may "
    "miss part of the structure, while complex models may mistake accidents for structure. Both errors hurt the "
    "ability to predict new observations, and much of this course is about how to balance them. "
)


def entropy_bits_per_byte(data: bytes) -> float:
    counts = Counter(data)
    n = len(data)
    return -sum((c / n) * math.log2(c / n) for c in counts.values())


def zlib_bits_per_byte(data: bytes) -> float:
    return 8 * len(zlib.compress(data, level=9)) / len(data)


N = 4000
sequences = {
    "repetitive 'AB...'": (b"AB" * N)[:N],
    "natural text": (TEXT.encode() * 10)[:N],
    "natural text (no repeats)": TEXT.encode(),
    "random bytes": os.urandom(N),
}

print(f"{'sequence':28s}{'bytes':>7s}{'H (bits/byte)':>15s}{'zlib (bits/byte)':>18s}")
for name, data in sequences.items():
    print(f"{name:28s}{len(data):7d}{entropy_bits_per_byte(data):15.3f}{zlib_bits_per_byte(data):18.3f}")

**What to notice:**

* **Random bytes** have ~8 bits/byte of entropy, and `zlib` cannot shrink them (it even adds overhead). There is no pattern to exploit.
* **Repetitive `ABAB...`**: the order-0 entropy says 1 bit/byte, but `zlib` does far better because it captures the *repetition*, a structure that symbol frequencies alone do not see.
  The true description ("repeat AB 2000 times") is tiny: this is the idea behind **Kolmogorov complexity**.
* **Natural text** sits in between: it has structure (letters, words, grammar), but also genuine information.
  The version repeated 10× compresses much better than the unique paragraph: the compressor *learned* the repetition.

## D3 — Model selection as compression (MDL)

Fit polynomials of degree $d = 0, \dots, 15$ to the $n = 50$ noisy points of D1. A polynomial of degree $d$ has $k = d + 1$ parameters.

Two-part code length (in bits, up to constants):

$$L(D, H) = \underbrace{\frac{k}{2}\log_2 n}_{L(H):\ \text{model}} + \underbrace{\frac{n}{2}\log_2 \text{MSE}_{\text{train}}}_{L(D \mid H):\ \text{residuals}}$$

This is the **Bayesian Information Criterion (BIC)** expressed in bits. It is a large-sample approximation: with very few points (try `sample(15)`) it becomes unreliable.

In [ ]:
degrees = np.arange(0, 16)
n = len(x_train)
train_err, test_err, mdl = [], [], []
for d in degrees:
    p = Polynomial.fit(x_train, y_train, deg=d)
    tr, te = mse(y_train, p(x_train)), mse(y_test, p(x_test))
    k = d + 1
    train_err.append(tr)
    test_err.append(te)
    mdl.append(k / 2 * np.log2(n) + n / 2 * np.log2(tr))

best_mdl, best_test = degrees[np.argmin(mdl)], degrees[np.argmin(test_err)]
print(f"MDL choice (training data only): d = {best_mdl:2d}   test MSE = {test_err[best_mdl]:.4f}")
print(f"Lowest test error (oracle)     : d = {best_test:2d}   test MSE = {test_err[best_test]:.4f}")
print(f"Most complex model             : d = 15   test MSE = {test_err[15]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(degrees, train_err, "o-", label="train MSE")
axes[0].semilogy(degrees, test_err, "s-", label="test MSE")
axes[0].axhline(SIGMA**2, color="gray", ls=":", label="noise floor $\\sigma^2$")
axes[0].axvline(best_mdl, color="tab:purple", ls="--", label=f"MDL choice (d={best_mdl})")
axes[0].set(xlabel="polynomial degree d", ylabel="MSE (log scale)", title="Fit vs. generalization")
axes[0].legend()

axes[1].plot(degrees, mdl, "o-", color="tab:purple")
axes[1].axvline(best_mdl, color="tab:purple", ls="--", label=f"MDL minimum (d={best_mdl})")
axes[1].set(xlabel="polynomial degree d", ylabel="description length (bits)", title="MDL / BIC")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, d, label in zip(axes, [1, best_mdl, 15], ["underfitting", "MDL choice", "overfitting"]):
    p = Polynomial.fit(x_train, y_train, deg=d)
    ax.scatter(x_train, y_train, color="k", s=15)
    ax.plot(grid, f(grid), "g--", alpha=0.6)
    ax.plot(grid, p(grid), color="tab:blue", lw=2)
    ax.set(title=f"d = {d} ({label})", ylim=(-2, 2), xlabel="x")
plt.tight_layout()
plt.show()

**Takeaway:** training error always decreases with complexity, so it cannot be used to choose a model.
The description length stops decreasing once extra parameters only encode noise.
MDL picks a model whose test error is close to the best one **without looking at test data**. It will not always pick the exact
oracle degree: with 50 noisy points, several degrees generalize almost equally well (notice how flat the test curve is in the middle).
Rerun with another random seed and watch the choice move while the test error stays low.

## Bridge — finding the model is an optimization problem

Above, `Polynomial.fit` solved for the parameters in closed form. In general there is no closed form: we must **search** the parameter space for the lowest loss.
Here is the loss landscape of the simplest model, $\hat{y} = w\,x + b$, on the same data.

In [ ]:
w_grid, b_grid = np.meshgrid(np.linspace(-6, 4, 200), np.linspace(-2, 3, 200))
pred = w_grid[..., None] * x_train + b_grid[..., None]
loss = np.mean((pred - y_train) ** 2, axis=-1)
i, j = np.unravel_index(loss.argmin(), loss.shape)

fig, ax = plt.subplots(figsize=(6, 4.5))
cs = ax.contourf(w_grid, b_grid, np.log(loss), levels=30, cmap="viridis")
ax.plot(w_grid[i, j], b_grid[i, j], "r*", ms=15, label=f"minimum: w={w_grid[i, j]:.2f}, b={b_grid[i, j]:.2f}")
fig.colorbar(cs, label="log MSE")
ax.set(xlabel="w", ylabel="b", title="Loss landscape of a linear model")
ax.legend()
plt.show()

A 200×200 grid search needed **40 000** loss evaluations for just 2 parameters.
With 1 000 parameters this is hopeless, which is why the next notebooks cover **blind (population-based)** and **gradient-based** optimization.